# Конспект. Модуль 12: Проблемы в production и их решения

**Курс:** Мини-курс RecSys (13 модулей)
**Модуль:** 12 из 13 — «Проблемы в production и их решения»
**Цель модуля:** перейти от «система построена и работает» (итог Модуля 11) к «система работает **правильно** в течение долгого времени, взаимодействуя с реальными людьми». Это единственный модуль курса, где основная сложность не в новой математике, а в системном, инженерном мышлении о том, как рекомендательная система **меняет** те самые данные, на которых она обучена, — и как с этим жить.

**Связь с предыдущими модулями:** каждая проблема этого модуля уже была как минимум один раз упомянута ранее (cold start — 1.5.1, popularity bias — 1.5.3, diversity — 8.7.2/11.5.2) — здесь мы даём этим проблемам полноценные, количественно обоснованные решения, а не просто констатируем их существование.

## 12.1 Холодный старт — практический playbook

### 12.1.1 Систематизация решений (сборка того, что уже было по частям)

| Тип cold start | Решение | Где уже разбиралось |
|:---|:---|:---:|
| Новый пользователь | Onboarding-опрос -> Content-Based фолбэк -> плавный переход на CF по мере накопления истории | 6.1.3, 7.2.2 |
| Новый товар | Контентные признаки (6) + временный «буст» видимости | 6.1.3, ниже 12.1.2 |
| Новая система целиком | Популярное / экспертные правила (Knowledge-Based, 1.2.4) | 1.2.4 |

### 12.1.2 «Буст» новых товаров — явный компромисс exploration/exploitation

Идея: временно **искусственно завышать** score нового товара в Ranking-стадии (Модуль 11.4) на фиксированный период (например, первые 2 недели или первые 1000 показов), чтобы гарантированно собрать достаточно данных о его реальной эффективности, прежде чем полагаться на модель, обученную практически без данных о нём. Формально это частный случай общей стратегии exploration, к которой мы вернёмся в разделе 12.2.3 — тот же принцип, применённый специально к товарам, а не к решениям модели в целом.

## 12.2 Popularity Bias

### 12.2.1 Механизм усиления (feedback loop) — полный проверенный численный пример

Возьмём 5 товаров с **почти одинаковой** начальной популярностью — у `I1` едва заметное преимущество перед `I2` (21 против 20 условных взаимодействий из общей истории в 76), `I3, I4, I5` заметно ниже. Правило рекомендации (упрощённая, но реалистичная модель): система показывает пользователям **только топ-2** товара по текущей популярности, при этом 1-я позиция забирает 65% кликов раунда, 2-я — 35% (реалистичное позиционное смещение — верхние позиции получают больше внимания, что вы уже фиксировали как независимую проблему в Модуле 8, обсуждая порядок в ранжировании).

**Динамика по раундам (реально просимулировано):**

| Раунд | I1 | I2 | I3 | I4 | I5 |
|:---:|:---:|:---:|:---:|:---:|:---:|
| 0 (старт) | 21.0 | 20.0 | 15.0 | 12.0 | 8.0 |
| 1 | 86.0 | 55.0 | 15.0 | 12.0 | 8.0 |
| 3 | 216.0 | 125.0 | 15.0 | 12.0 | 8.0 |
| 5 | 346.0 | 195.0 | 15.0 | 12.0 | 8.0 |
| 7 | 476.0 | 265.0 | 15.0 | 12.0 | 8.0 |

**Ключевое наблюдение:** товары `I3, I4, I5` **не получили ни единого нового взаимодействия за все 7 раундов** — они попросту никогда не попали в топ-2 и стали структурно невидимыми для системы. При этом начальный разрыв между `I1` и `I2` был минимальным (`21` против `20`, всего 2.4%) — но позиционное смещение раунд за раундом **усиливало** этот крошечный начальный перевес: к раунду 7 `I1` (476) почти вдвое обходит `I2` (265). К концу симуляции `I1+I2` забирают **95.5%** всего трафика.

**Количественная мера концентрации (коэффициент Джини):**

In [ ]:
Gini НАЧАЛЬНЫЙ (раунд 0) = 0.1789
Gini ПОСЛЕ 7 раундов      = 0.6129

Рост коэффициента Джини более чем втрое за короткий период — количественное, объективное подтверждение того, что система **сама** создаёт и усиливает неравенство в видимости товаров, причём отправной точкой послужило почти незначимое начальное различие.

### 12.2.2 Почему это опасно не только «этически», но и с точки зрения бизнеса

Помимо очевидных проблем разнообразия (Модуль 1.5.3), это прямая угроза для **самой системы обучения**: если `I3, I4, I5` никогда не показываются, о них никогда не собирается новая статистика взаимодействий — модель никогда не сможет узнать, понравились бы они пользователям **лучше**, чем `I1/I2`, потому что у неё просто не появится для этого данных. Система застревает в локальном оптимуме, из которого не способна выбраться самостоятельно.

### 12.2.3 Решение — Exploration через ε-greedy

**Идея:** с небольшой вероятностью `ε` показывать не «лучший по текущей модели» вариант, а случайный/менее уверенный — специально ради сбора новых данных, а не ради немедленной максимизации клика.

In [ ]:
epsilon = 0.1
При 10 000 рекомендаций ожидается ~1000 "исследовательских" показов
Остальные ~9000 -- обычные, "жадные" (по текущей лучшей модели)

Это прямая и простая тактическая защита от механизма из раздела 12.2.1 — гарантирует, что даже структурно «невидимые» `I3, I4, I5` периодически всё же получают шанс показаться и накопить хоть какую-то статистику.

### 12.2.4 Inverse Propensity Scoring (IPS) — коррекция смещения уже собранных логов

Отдельная от exploration проблема: даже если мы **исправим** будущие рекомендации через ε-greedy, исторические логи, накопленные **до** этого исправления, уже искажены — популярные товары в них систематически «переоценены» просто потому, что были показаны чаще (больше шансов получить клик просто по объёму показов), а не обязательно потому, что были объективно лучше.

**Формула IPS:**

In [ ]:
IPS-оценка ценности товара = среднее(reward_i / propensity_i)

где `propensity_i` — вероятность того, что товар вообще был **показан** (залогирован) в исторических данных.

**Полный проверенный численный пример.** `I1` показывался часто (`propensity=0.50`) и получил `1` клик из `3` показов. `I5` показывался редко (`propensity=0.05`) и получил `1` клик из `1` показа:

In [ ]:
Наивная (сырая) оценка ценности I1 = среднее([1,0,0]) = 0.3333
IPS-скорректированная оценка I1 = среднее([1/0.50, 0/0.50, 0/0.50]) = 0.6667
IPS-скорректированная оценка I5 = среднее([1/0.05]) = 20.0000

**Интерпретация:** IPS резко «повышает вес» результата, полученного от редко показываемого товара — логика в том, что если товар показывался всего в 5% случаев, но всё равно принёс клик, это **более сильный** сигнал качества на единицу воздействия, чем клик от товара, который и так показывался в 10 раз чаще. IPS — стандартный инструмент офлайн-оценки (off-policy evaluation) при использовании исторических, уже смещённых логов, широко применяемый именно там, где нельзя провести чистый A/B-тест немедленно (следующий раздел).

## 12.3 Serendipity и Diversity — интеграция с уже освоенным инструментарием

Этот раздел не вводит новую математику — он явно связывает уже освоенные вами инструменты (Diversity как метрика, Модуль 8.7.2; MMR как алгоритм re-ranking, Модуль 11.5.2) с общей темой production-рисков этого модуля. Popularity Bias (12.2) — это, по сути, частный случай общей проблемы низкой Diversity **на уровне всей системы**, а не только внутри одного списка. Тактика exploration (12.2.3) решает проблему на уровне **обучающих данных**, MMR (11.5.2) — на уровне **финального списка**. Обе тактики дополняют друг друга и обычно применяются одновременно в реальных production-системах: exploration гарантирует, что у модели вообще есть шанс узнать о «длинном хвосте», MMR гарантирует, что даже при уверенном знании модели о лучших вариантах пользователю не покажут «пять одинаковых».

## 12.4 A/B тестирование

### 12.4.1 Почему офлайн-метрик (Модуль 8) недостаточно

Офлайн NDCG/Precision@K считаются на **исторических** данных — а эти данные, как мы только что увидели в разделе 12.2.1, уже искажены поведением **предыдущей** версии модели (position bias, exposure bias). Высокий NDCG на таком историческом тесте не гарантирует роста реальных бизнес-метрик после развёртывания — единственный способ получить непредвзятый ответ — показать **разным** случайно выбранным пользователям **разные** версии системы одновременно, в реальном времени.

### 12.4.2 Дизайн эксперимента — ключевые решения

- **Случайное разделение пользователей** на контрольную (старая система) и тестовую (новая) группы — критично именно **случайное**, а не, например, «по регионам» или «по времени суток» (это внесло бы систематическое смещение).
- **Выбор основной метрики заранее**, до начала эксперимента (иначе возникает соблазн выбрать ту метрику постфактум, которая «случайно» показала улучшение — классическая ошибка p-hacking).
- **Контроль длительности** — слишком короткий эксперимент подвержен «эффекту новизны» (novelty effect: пользователи реагируют на **любое** изменение интерфейса первые дни, даже если оно не улучшает предпочтения по существу).

### 12.4.3 Полный проверенный численный пример — проверка статистической значимости

Control-группа (старая система): `500` кликов из `10 000` показов -> `CTR = 5.00%`.
Treatment-группа (новая система, например, с добавленным MMR-переранжированием из Модуля 11.5.2): `550` кликов из `10 000` показов -> `CTR = 5.50%`.

**Two-proportion z-test:**

In [ ]:
p_control = 0.0500
p_treatment = 0.0550
Относительный прирост = +10.00%

z-statistic = 1.5852
p-value = 0.1129

**Вывод:** при стандартном пороге `α=0.05`, `p-value=0.1129 > 0.05` — **наблюдаемое различие статистически НЕ значимо**, несмотря на то, что относительный прирост CTR выглядит внушительно (`+10%`). Это критически важный практический урок: заметное на глаз улучшение может оказаться в пределах случайного шума при недостаточном размере выборки. Правильное инженерное решение здесь — не спешить с выводами, а либо увеличить размер выборки/длительность теста, либо честно признать результат неубедительным.

### 12.4.4 Ключевые онлайн-метрики

`CTR`, конверсия, среднее время сессии, `retention` (удержание пользователей за более длинный горизонт, что особенно важно для отслеживания «эффекта новизны» из раздела 12.4.2 — если прирост CTR исчезает через неделю, а `retention` не меняется, скорее всего, это и был novelty effect, а не реальное улучшение).

## 12.5 Онлайн vs Офлайн рекомендации

### 12.5.1 Сравнительная таблица

| | Офлайн | Онлайн | Гибрид (типичный production) |
|:---|:---|:---|:---|
| Когда считаются предсказания | Заранее (batch), Модуль 4.4.1 | В момент запроса | Базовый набор офлайн + лёгкая онлайн-коррекция |
| Актуальность | Может устареть | Максимально свежо | Компромисс |
| Latency | Мгновенно (lookup) | Зависит от инфраструктуры (Модуль 11.1) | Определяется онлайн-частью |
| Инфраструктурная сложность | Низкая | Высокая | Средняя |

### 12.5.2 Практический компромисс

Большинство крупных production-систем используют именно гибридную схему: базовый список рекомендаций (Retrieval+Ranking, Модуль 11) предвычисляется офлайн и кэшируется, но затем **дополняется или незначительно переранжируется** онлайн с учётом самых последних действий пользователя (например, товаров, добавленных в корзину минуту назад) — сочетая низкую latency офлайн-подхода с достаточной свежестью для того, чтобы не выглядеть «неактуальным».

## 12.6 Практика

### 12.6.1 Реализация ε-greedy обёртки над готовым ранкером

In [ ]:
import numpy as np

def epsilon_greedy_rerank(ranked_candidates: list, all_candidates: list, epsilon: float = 0.1):
    """
    ranked_candidates: уже отсортированный список от Ranking-стадии (Модуль 11.4)
    all_candidates: полный пул кандидатов (включая "холодные", редко показываемые)
    """
    if np.random.random() < epsilon:
        # Exploration: случайный кандидат вместо "жадного" топа (раздел 12.2.3)
        exploratory_pick = np.random.choice(all_candidates)
        return [exploratory_pick] + [c for c in ranked_candidates if c != exploratory_pick]
    return ranked_candidates  # обычный "жадный" режим

### 12.6.2 IPS-оценка на исторических логах

In [ ]:
def ips_estimate(logs: list) -> dict:
    """logs: список словарей {'item': ..., 'propensity': ..., 'reward': ...} (раздел 12.2.4)."""
    from collections import defaultdict
    ips_values = defaultdict(list)
    for entry in logs:
        ips_values[entry['item']].append(entry['reward'] / entry['propensity'])
    return {item: np.mean(vals) for item, vals in ips_values.items()}

### 12.6.3 Проверка значимости A/B-теста

In [ ]:
from scipy import stats
import numpy as np

def ab_test_significance(n_control, clicks_control, n_treatment, clicks_treatment, alpha=0.05):
    p_c, p_t = clicks_control/n_control, clicks_treatment/n_treatment
    p_pool = (clicks_control+clicks_treatment)/(n_control+n_treatment)
    se = np.sqrt(p_pool*(1-p_pool)*(1/n_control+1/n_treatment))
    z = (p_t - p_c) / se
    p_value = 2*(1-stats.norm.cdf(abs(z)))
    return {'z': z, 'p_value': p_value, 'significant': p_value < alpha}

### 12.6.4 Практическое задание

- Реализовать полную симуляцию из раздела 12.2.1 (feedback loop) самостоятельно, поэкспериментировать с параметром позиционного смещения (65%/35% -> попробовать 90%/10%) — как быстро при более жёстком смещении растёт Gini-коэффициент?
- На реальных логах MovieLens (используя временные метки как прокси для «раундов показа») оценить фактический коэффициент Джини по распределению числа оценок на фильм — сравнить с игрушечным примером курса.
- Рассчитать необходимый размер выборки для A/B-теста (`statsmodels.stats.power.NormalIndPower`), чтобы надёжно (при `power=0.8`) обнаружить прирост CTR с `5.0%` до `5.5%`, аналогичный примеру 12.4.3 — и сравнить с уже использованными `10 000` наблюдениями на группу: было ли их вообще достаточно?

### 12.6.5 Вопросы для самопроверки

1. В разделе 12.2.1 крошечное начальное различие (`21` против `20`) превратилось в почти двукратный разрыв. Если бы начальные значения `I1` и `I2` были абсолютно **равны** (`20.5` и `20.5`), что определило бы, какой из них в итоге «выиграет» гонку за топ-позицию при такой симуляции? Что это говорит о роли случайности в реальных production-системах?
2. Объясните, почему IPS-оценка (12.2.4) может давать **очень большие** числа для крайне редко показываемых товаров (в примере — `20.0`), и почему это одновременно и полезное свойство (честная коррекция), и источник статистической нестабильности (высокая дисперсия оценки).
3. В примере A/B-теста (12.4.3) прирост CTR составил `+10%` относительно, но оказался статистически незначим. Опишите, как изменился бы `p-value`, если бы тот же самый относительный прирост (`+10%`) наблюдался при вдесятеро большем размере каждой группы (`100 000` вместо `10 000`) — какая величина в формуле z-statistic отвечает за эту чувствительность к размеру выборки?
4. Почему exploration (12.2.3) и MMR (11.5.2) решают **разные** аспекты одной и той же общей проблемы (низкое разнообразие/предвзятость системы), и почему production-система обычно нуждается в обоих механизмах одновременно, а не в одном из них?

## Глоссарий модуля 12

| Термин | Короткое определение |
|:---|:---|
| Cold Start Boost | Временное искусственное завышение score новых товаров для сбора данных |
| Feedback Loop | Самоусиливающийся цикл: показ -> клики -> ещё больше показов |
| Position Bias | Верхние позиции выдачи получают непропорционально больше внимания |
| Коэффициент Джини | Мера концентрации/неравенства распределения (0=равномерно, 1=максимально неравномерно) |
| ε-greedy | Стратегия: с вероятностью ε — исследование, иначе — использование лучшего известного варианта |
| IPS (Inverse Propensity Scoring) | Коррекция смещения логов делением reward на вероятность показа |
| Propensity | Вероятность того, что объект вообще был показан/залогирован |
| Novelty Effect | Временный всплеск метрик из-за самого факта изменения, не связанный с реальным улучшением |
| Off-policy evaluation | Оценка новой стратегии по логам, собранным при старой стратегии |

**Связь со следующим модулем:** мы прошли полный путь от «что такое рекомендация» (Модуль 1) до понимания всех практических рисков production-эксплуатации (этот модуль). Модуль 13 — финальный капстоун курса: полноценный end-to-end проект, в котором Retrieval (Модули 5, 10, 11), Ranking (Модуль 9, 11), Re-ranking (Модуль 11.5) и защита от проблем этого модуля (exploration, мониторинг) собираются в единый, готовый к портфолио репозиторий.